# ONI SPT Pipeline
Notebook-first setup, 50-frame pilot, QC review, full-batch execution, and condition-level diffusion reporting. All numerical work is implemented in the ordered `stepXX_*.py` scripts beside this notebook.

## 1. Editable parameters
Run the pilot first. Change `run_full_batch` only after the MP4 and 1200-dpi trajectory overlay pass visual review.

In [ ]:
from pathlib import Path
import os

config_path = Path('configs/fvp-validation.yaml')
full_config_path = Path('configs/fvp-full.yaml')
pilot_max_files = 1
pilot_max_frames = 50
reuse_complete_outputs = True
overwrite_stage_outputs = False
run_full_batch = os.environ.get('ONI_SPT_RUN_FULL_BATCH', '0') == '1'

print(f'Config: {config_path.resolve()}')
print(f'Full batch enabled: {run_full_batch}')

## 2. Preflight
This cell scans metadata and reports accepted/rejected TIFFs without processing image pages.

In [ ]:
from IPython.display import Image, Markdown, Video, display
from spt_shared import output_dirs
from step01_inspect_inputs import inspect_inputs
from run_ONI_SPT import run_pipeline

cfg, manifest = inspect_inputs(config_path)
display(manifest[['filename', 'condition', 'actual_frames', 'width', 'pixel_size_um', 'frame_interval_s', 'status', 'reason', 'warnings']])
print(f"Accepted: {(manifest.status == 'accepted').sum()}; unused: {(manifest.status != 'accepted').sum()}")

## 3. Run the 50-frame pilot

In [ ]:
pilot = run_pipeline(
    str(config_path), max_files=pilot_max_files, max_frames=pilot_max_frames,
    resume=reuse_complete_outputs, force=overwrite_stage_outputs, through_report=True,
)
print('Pilot complete')

## 4. Review segmentation and frame-resolved QC

In [ ]:
dirs = output_dirs(cfg['output_dir'])
for path in sorted(dirs['segmentation'].glob('*__segmentation_overlay.png')):
    display(Markdown(f'### {path.name}'))
    display(Image(filename=str(path)))
for path in sorted((dirs['qc'] / 'detection_videos').glob('*.mp4')):
    display(Markdown(f'### Spot detection: {path.name}'))
    display(Video(filename=str(path), embed=True))
for path in sorted((dirs['qc'] / 'trajectory_videos').glob('*.mp4')):
    display(Markdown(f'### Trajectories: {path.name}'))
    display(Video(filename=str(path), embed=True))
for path in sorted((dirs['qc'] / 'trajectory_overlays').glob('*1200dpi.png')):
    print('Full-resolution trajectory overlay:', path)

## 5. Review condition-level diffusion report

In [ ]:
summary = dirs['reports'] / 'condition_summary.csv'
if summary.exists():
    import pandas as pd
    display(pd.read_csv(summary))
print('PDF report:', dirs['reports'] / 'diffusion_comparison_report.pdf')

## 6. Full batch
This remains disabled until the corrected float32 bandpass and QC videos are approved.

In [ ]:
if run_full_batch:
    full = run_pipeline(str(full_config_path), resume=True, through_report=True)
    print('Full batch complete')
else:
    print('Pilot only. Launch with ONI_SPT_RUN_FULL_BATCH=1 after QC approval.')